# Regression Baseline — Multi-Model Comparison

Predicts **remaining case time** (hours) using Ridge, RandomForest, and HistGradientBoosting.
Features: `BasicControlFlowFeatures` only (log-based; time-series features excluded).
All run parameters are defined in `configs/regression.yaml`.

In [1]:
import contextlib
import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from tqdm import tqdm

from spi_time_series.config import load_config
from spi_time_series.config.loader import build_estimator
from spi_time_series.data.schemas import ModelArtifact, RawData
from spi_time_series.evaluation.metrics import evaluate
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
from spi_time_series.models.trainer import train
from spi_time_series.pipeline import PipelineBuilder
from spi_time_series.pipeline.checkpointing import checkpoint

logging.basicConfig(level=logging.INFO, force=True)

## Config

In [2]:
config = load_config(Path("../configs/regression.yaml"))

MODEL_CONFIGS = {
    name: (build_estimator(m), m.param_grid)
    for name, m in config.models.items()
}

## Target generator

`TargetGenerator` has signature `(trace, start_idx, end_idx) -> float` — no `col_idx` parameter.
`_col_idx_store` is populated from `preprocessed.col_idx` after the split and read by
`remaining_time_hours` during feature extraction. Safe because stages run sequentially.

In [3]:
_col_idx_store: dict[str, int] = {}


def remaining_time_hours(trace, start_idx: int, end_idx: int) -> float:
    ts_idx = _col_idx_store["time:timestamp"]
    delta = trace[-1, ts_idx] - trace[end_idx - 1, ts_idx]
    return delta.total_seconds() / 3600

## Pipeline

In [4]:
pipeline = (
    PipelineBuilder.from_config(config)
    .with_feature_extractor(
        extract_features_builder(
            [
                BasicControlFlowFeatures(
                    one_hot_encode_categorical=config.features.one_hot_encode_categorical,
                )
            ],
            remaining_time_hours,
        )
    )
    .build()
)

INFO:spi_time_series.data.dataset:Dataset found at G:\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
g:\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


## Stage 1 — Load and clean

In [5]:
raw = RawData(event_log=pipeline.dataset.log)

cleaned = checkpoint(
    config.checkpoint_dir / "cleaned_log.pkl",
    lambda: pipeline.preprocessor(raw),
    params=config.checkpoint_params(),
)

INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\cleaned_log__0d2cf5a9.pkl
INFO:spi_time_series.preprocessing.preprocess:Filtering for valid end activities: ['W_Validate application', 'W_Call after offers', 'W_Call incomplete files', 'O_Cancelled', 'A_Denied']
INFO:spi_time_series.preprocessing.preprocess:Remaining events after cleaning: 1193604
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\cleaned_log__0d2cf5a9.pkl


## Stage 2 — Split and extract features

Splitting is not checkpointed — it is fast and produces lazy iterators that cannot be serialised.
The column-index store is populated from `preprocessed.col_idx` so that `remaining_time_hours`
can look up the timestamp column index during feature extraction.

In [6]:
preprocessed = pipeline.splitter(cleaned)
_col_idx_store.update(preprocessed.col_idx)

features = checkpoint(
    config.checkpoint_dir / "features.pkl",
    lambda: pipeline.feature_extractor(preprocessed),
    params=config.checkpoint_params(),
)

print(f"Train shape: {features.X_train.shape}")
print(f"Test shape:  {features.X_test.shape}")

INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.
INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\features__0d2cf5a9.pkl
Processing cases: 100%|██████████| 6247/6247 [00:26<00:00, 237.69it/s]
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\features__0d2cf5a9.pkl


Train shape: (822706, 33)
Test shape:  (227073, 33)


## Stage 3 — Hyperparameter search and fit

Randomised CV search (`refit=False`) finds best hyperparameters per model, then `train()` fits
each best estimator inside a full preprocessing pipeline on the complete training set.
The artifact is checkpointed so this stage is skipped on re-runs when the config is unchanged.

In [7]:
@contextlib.contextmanager
def _tqdm_joblib(pbar):
    class _CB(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *a, **kw):
            pbar.update(self.batch_size)
            return super().__call__(*a, **kw)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _CB
    try:
        yield pbar
    finally:
        joblib.parallel.BatchCompletionCallBack = old


def _fit_artifact() -> ModelArtifact:
    search_pre = ColumnTransformer(
        [
            (
                "num",
                SimpleImputer(strategy="median"),
                make_column_selector(dtype_include=np.number),
            )
        ],
        remainder="drop",
    )
    X_num = search_pre.fit_transform(features.X_train)
    n_fits = config.search.n_iter * config.search.cv_folds

    best_estimators = {}
    for name, (base_est, hp_grid) in MODEL_CONFIGS.items():
        search = RandomizedSearchCV(
            base_est,
            param_distributions=hp_grid,
            n_iter=config.search.n_iter,
            cv=config.search.cv_folds,
            scoring="neg_mean_absolute_error",
            refit=False,
            random_state=config.search.random_state,
        )
        with _tqdm_joblib(tqdm(desc=name, total=n_fits)):
            search.fit(X_num, features.y_train)
        print(
            f"{name}: best={search.best_params_}  CV MAE={-search.best_score_:.2f} h"
        )
        best_estimators[name] = clone(base_est).set_params(
            **search.best_params_
        )

    return train(features, best_estimators)


artifact = checkpoint(
    config.checkpoint_dir / "artifact.pkl",
    _fit_artifact,
    params=config.checkpoint_params(),
)

INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\artifact__0d2cf5a9.pkl
ridge:   0%|          | 0/30 [00:00<?, ?it/s]

ridge: best={'alpha': 1000.0}  CV MAE=183.76 h


ridge:   0%|          | 0/30 [00:09<?, ?it/s]


random_forest: best={'n_estimators': 150, 'min_samples_split': 2, 'max_features': 0.5, 'max_depth': 20}  CV MAE=176.22 h


hist_gradient_boosting: 900it [01:18, 13.74it/s]                      INFO:spi_time_series.models.trainer:Training model: ridge


hist_gradient_boosting: best={'max_iter': 200, 'max_depth': None, 'learning_rate': 0.2}  CV MAE=176.52 h


INFO:spi_time_series.models.trainer:Model ridge trained.
INFO:spi_time_series.models.trainer:Training model: random_forest
hist_gradient_boosting: 930it [01:33, 13.74it/s]INFO:spi_time_series.models.trainer:Model random_forest trained.
INFO:spi_time_series.models.trainer:Training model: hist_gradient_boosting
INFO:spi_time_series.models.trainer:Model hist_gradient_boosting trained.
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\artifact__0d2cf5a9.pkl


## Stage 4 — Evaluate

In [8]:
report = evaluate(artifact, features)

INFO:spi_time_series.evaluation.metrics:Evaluating model: ridge
INFO:spi_time_series.evaluation.metrics:Model ridge evaluated across 8 prefix lengths.
INFO:spi_time_series.evaluation.metrics:Evaluating model: random_forest
INFO:spi_time_series.evaluation.metrics:Model random_forest evaluated across 8 prefix lengths.
INFO:spi_time_series.evaluation.metrics:Evaluating model: hist_gradient_boosting
INFO:spi_time_series.evaluation.metrics:Model hist_gradient_boosting evaluated across 8 prefix lengths.


## Results

MAE, RMSE (hours), and R² per model and prefix length.

In [9]:
pd.DataFrame(
    [
        {
            "model": model,
            "prefix_length": pl,
            "MAE (h)": round(report.metrics[model][pl]["mae"], 3),
            "RMSE (h)": round(report.metrics[model][pl]["rmse"], 3),
            "R²": round(report.metrics[model][pl]["r2"], 4),
        }
        for model in sorted(report.model_names)
        for pl in report.prefix_lengths
    ]
)

,model,prefix_length,MAE (h),RMSE (h),R²
0,hist_gradient_boosting,3,248.798,310.368,-0.1708
1,hist_gradient_boosting,4,240.186,283.638,0.0214
2,hist_gradient_boosting,5,239.879,283.864,0.0198
3,hist_gradient_boosting,6,239.831,283.585,0.0231
4,hist_gradient_boosting,7,237.772,281.960,0.0218
5,hist_gradient_boosting,8,237.249,280.587,0.0313
6,hist_gradient_boosting,9,235.506,279.364,0.0297
7,hist_gradient_boosting,10,157.198,219.712,0.3859
8,random_forest,3,240.494,283.130,0.0257
9,random_forest,4,241.647,285.333,0.0096


### MAE by prefix length (pivot)

Rows = prefix length, columns = model.

In [10]:
pd.DataFrame(
    {
        model: {
            pl: round(report.metrics[model][pl]["mae"], 1)
            for pl in report.prefix_lengths
        }
        for model in sorted(report.model_names)
    }
).rename_axis("prefix_length")

,hist_gradient_boosting,random_forest,ridge
prefix_length,,,
3,248.8,240.5,242.6
4,240.2,241.6,242.1
5,239.9,241.0,241.2
6,239.8,240.1,241.8
7,237.8,238.2,241.2
8,237.2,237.8,241.0
9,235.5,237.2,240.3
10,157.2,157.1,164.2
